# Day 1

In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [2]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1285
Evidently documents: 95


In [3]:
learning_data_engineering_docs = read_repo_data('Joyan9', 'learning_data_engineering')
print(f"LDE documents: {len(learning_data_engineering_docs)}")

LDE documents: 47


# Day 2

In [4]:
len(evidently_docs[45]['content'])

21712

### Sliding Window Chunking

The content field is 21,712 characters long. The simplest thing we can do is cut it into pieces of equal length. For example, for size of 2000 characters

However, this approach has disadvantages:

- Context loss: Important information might be split in the middle
- Incomplete sentences: Chunks might end mid-sentence
- Missing connections: Related information might end up in different chunks

That's why, in practice, we usually make sure there's overlap between chunks. For size 2000 and overlap 1000, we will have:

- Chunk 1: 0..2000
- Chunk 2: 1000..3000
- Chunk 3: 2000..4000

**This approach is called the sliding window chunking**

In [5]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [6]:
evidently_chunks = []

for doc in evidently_docs:
    # create a shollow copy of each doc
    doc_copy = doc.copy()
    # the pop method removes the content key and returns the value
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, size=2000, step=1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)


In [7]:
len(evidently_chunks)

576

In [6]:
# splitting by paragraphs and sections

import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [7]:
evidently_chunks_split_by_level = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks_split_by_level.append(section_doc)

len(evidently_chunks_split_by_level)

266

In [8]:
import os

from groq import Groq

client = Groq(
    # This is the default and can be omitted
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in today's technology landscape, and their importance can be attributed to several factors:

1. **Efficient Processing**: Fast language models can process and analyze vast amounts of text data quickly, making them ideal for applications where speed is essential, such as real-time chatbots, virtual assistants, and language translation systems.
2. **Improved User Experience**: By providing rapid responses to user queries, fast language models can significantly enhance the user experience. This is particularly important in customer service, where timely responses can lead to increased customer satisfaction and loyalty.
3. **Increased Productivity**: Fast language models can automate many tasks, such as text summarization, sentiment analysis, and content generation, allowing humans to focus on higher-level tasks that require creativity, critical thinking, and problem-solving skills.
4. **Competitive Advantage**: Organizations that leverage fast language mod

In [13]:
# chunking with a LLM - useful when the documentation is not structured, the simpler methods do not provide adequate results, and you have a budget

prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

import os

from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)


def llm(prompt, model="meta-llama/llama-4-scout-17b-16e-instruct"):

    chat_completion = client.chat.completions.create(messages=[
                                                                {
                                                                    "role": "system",
                                                                    "content": "You are a helpful assistant."
                                                                },
                                                                {
                                                                    "role": "user",
                                                                    "content": prompt,
                                                                }
                                                                ],
                                                                model= model,
                                                                )

    output_text = chat_completion.choices[0].message.content

    return output_text
    

def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [14]:
from tqdm.auto import tqdm
# tqdm is a library that shows progress bars. It helps you track progress when processing a large number of documents.

evidently_chunks_split_by_llm = []

# processing only the first 45 documents to avoid rate limit error
for doc in tqdm(evidently_docs[:45]):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks_split_by_llm.append(section_doc)

  0%|          | 0/45 [00:00<?, ?it/s]

In [16]:
evidently_chunks_split_by_llm_2 = []

for doc in tqdm(evidently_docs[45:]):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks_split_by_llm_2.append(section_doc)

  0%|          | 0/50 [00:00<?, ?it/s]

In [15]:
len(evidently_chunks_split_by_llm)

155

In [17]:
len(evidently_chunks_split_by_llm_2)

183

# Day 3 - Search

### 1. Lexical Search / Text Search
We look for exact matches between our query and the documents.


In [22]:
from minsearch import Index

# We create an index that will search through four text fields: chunk content, title, description, and filename
index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)

<frozen abc>:107: RuntimeWarning: coroutine 'AbstractAgent.run' was never awaited


NameError: name 'evidently_chunks' is not defined

In [ ]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)
print(results)

In [23]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')

de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

faq_index.search('Can I still join the course after the start date?')

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '9e508f2212',
  'question': 'Course: When does the course start?',
  'sort_order': 1,
  'content': "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\n- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp).\n- Don’t forget to register in DataTalks.Club's Slack and [join](htt

***Text search would fail for queries where the words used do not match the content***

### 2. Vector Search

Coverts words and phrases into vectors (embeddings) and then compares if there are semantically similar or not

In [8]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)

In [17]:
similarity = v_query.dot(v_doc)

similarity

np.float32(0.5190934)

*The dot product measures similarity between vectors*

Values closer to 1 indicate higher similarity, closer to 0 means lower similarity. This works because the model creates normalized embeddings where cosine similarity equals the dot product.

So we can create embeddings for all documents, then compute similarity between the query and each document to find the most similar ones.


In [18]:
from tqdm.auto import tqdm
import numpy as np
from minsearch import VectorSearch

faq_embeddings = []

for d in tqdm(de_dtc_faq):
    text = d['question'] + ' ' + d['content']
    v = embedding_model.encode(text)
    faq_embeddings.append(v)

faq_embeddings = np.array(faq_embeddings)


faq_vindex = VectorSearch()
# This creates a vector search index using our embeddings and original documents
faq_vindex.fit(faq_embeddings, de_dtc_faq)

  0%|          | 0/498 [00:00<?, ?it/s]

In [19]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)


In [20]:
results

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '068529125b',
  'question': 'Course - Can I follow the course after it finishes?',
  'sort_order': 8,
  'content': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.',
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/008_068529125b_course-can-i-follow-the-course-after-i

### 3. Hybrid Search

- Text Search - fast, efficient but misses queries which are paraphrased but are semantically similar
- Vector Search - Needs a vector embedding model, captures semantically similar queries but misses exact keyword matches

#### A Quick Example
| Query | Vector Search Result | Keyword Search Result |
| :--- | :--- | :--- |
| **"Acme Drill 500"** | Finds "Heavy-duty power tools" | Finds "Acme Drill 500" |
| **"Tool for making holes"** | Finds "Acme Drill 500" | Finds nothing (no match for "holes") |

**By combining them, you get the best of both worlds.**

In [21]:
query = 'Can I join the course now?'

text_results = faq_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = faq_vindex.search(q, num_results=5)

final_results = text_results + vector_results
final_results

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '9e508f2212',
  'question': 'Course: When does the course start?',
  'sort_order': 1,
  'content': "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\n- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp).\n- Don’t forget to register in DataTalks.Club's Slack and [join](htt

In [4]:
def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results


# Day 4 - Building an Agent
Now it's time to create an AI agent that will use this data through the search engine that we created yesterday.
This allows us to build context-aware agents. They can provide accurate, relevant answers based on your specific domain knowledge rather than just general training data.



### Asking a Question to the LLM without Search Tool

In [32]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=chat_messages,
)

print(response.choices[0].message.content)

I need a bit more information. What course are you referring to? Is it an online course, a university course, or something else? Additionally, when does the course start or is it already in progress? This will help me provide a more accurate answer to your question.


In [34]:
import json

def text_search(query):
    return faq_index.search(query, num_results=5)
    
# Updated Tool Definition for Groq
text_search_tool = {
    "type": "function",
    "function": {
        "name": "text_search",
        "description": "Search the course FAQ database.", # Keep it short and clear
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query"
                }
            },
            "required": ["query"]
        }
    }
}

system_prompt = """
You are a helpful assistant for a course.
You have access to a tool called 'text_search'.

RULES:
1. Always use 'text_search' to find answers before responding to user questions.
2. If you use a tool, do not provide any conversational text until after you receive the tool results.
3. Be concise.
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

# 1. First Call: Agent decides to use the tool
response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=chat_messages,
    tools=[text_search_tool],
    tool_choice="auto"
)

message = response.choices[0].message
print(f"Message: {message}")

tool_calls = message.tool_calls

if tool_calls:
    # 2. Execute the function
    # Groq can suggest multiple tool calls; here we handle the first one
    chat_messages.append(message) # Add the assistant's call to history
    
    for tool_call in tool_calls:
        function_args = json.loads(tool_call.function.arguments)
        result = text_search(query=function_args.get("query"))
        
        # 3. Add the tool result to the conversation
        chat_messages.append({
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": "text_search",
            "content": json.dumps(result),
        })

    # 4. Second Call: Agent generates final answer using search results
    final_response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=chat_messages
    )
    print("Final Response: " ,final_response.choices[0].message.content)

Message: ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='8r0m4jqrq', function=Function(arguments='{"query":"join course now"}', name='text_search'), type='function')])
Final Response:  Yes, you can still join the course after the start date. You're still eligible to submit the homework, but be aware of the deadlines for turning in homework and the final project.


In [35]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.
    Args:
        query: The search query string.
    """
    return faq_index.search(query, num_results=5)

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel
from typing import List, Any

# 1. Define the model
model = GroqModel('llama-3.1-8b-instant')

# 2. Match the prompt name to the function name
system_prompt = """
You are a helpful course assistant. 
You have access to a 'text_search' tool. 
Always use this tool to look up information before answering questions.
"""

agent = Agent(
    model,
    system_prompt=system_prompt,
)

# 3. Rename this to match the prompt ('text_search')
@agent.tool_plain
def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQs index.
    Args:
        query: The search query string.
    """
    return faq_index.search(query, num_results=5)

# 4. Run the agent
# In a notebook, use 'await'
result = await agent.run("I just discovered the course, can I join now?")

# 5. Use .data to see the final natural language answer
print(result.output)

It seems like this course starts and ends with open enrollment until at some point of Oct.  This suggests an open enrollment period that could be until or past the 15th in October. Please check text_search for the exact dates of the open enrollment period and the course start and end dates.


In [43]:
result.all_messages_json()

b'[{"parts":[{"content":"\\nYou are a helpful course assistant. \\nYou have access to a \'text_search\' tool. \\nAlways use this tool to look up information before answering questions.\\n","timestamp":"2026-03-26T17:09:24.869097Z","dynamic_ref":null,"part_kind":"system-prompt"},{"content":"I just discovered the course, can I join now?","timestamp":"2026-03-26T17:09:24.869101Z","part_kind":"user-prompt"}],"timestamp":"2026-03-26T17:09:24.869334Z","instructions":null,"kind":"request","run_id":"080a8b47-5c95-487a-b169-7e4e207d95af","metadata":null},{"parts":[{"tool_name":"text_search","args":"{\\"query\\":\\"Can I join the course if I just discovered it?\\"}","tool_call_id":"c8djb05kt","id":null,"provider_name":null,"provider_details":null,"part_kind":"tool-call"}],"usage":{"input_tokens":266,"cache_write_tokens":0,"cache_read_tokens":0,"output_tokens":23,"input_audio_tokens":0,"cache_audio_read_tokens":0,"output_audio_tokens":0,"details":{}},"model_name":"llama-3.1-8b-instant","timestamp

In Pydantic AI, all_messages_json() returns the complete history of Requests (what we sent to the LLM) and Responses (what the LLM sent back).

This output is essentially the **"Black Box" recording** of your Agent's entire thought process. It shows exactly how the conversation evolved from the first prompt to the final answer.

In Pydantic AI, `all_messages_json()` returns the complete history of **Requests** (what we sent to the LLM) and **Responses** (what the LLM sent back).

### 1. The Initial Setup (Request)
The first block shows the **System Prompt** and your **User Prompt** being bundled together and sent to Groq. 
* **Key Part:** `part_kind: "system-prompt"` and `user-prompt`.
* This is the "Start" state.

### 2. The First "Aha!" (Response -> Tool Call)
The model (Llama 3.1) realizes it doesn't know the answer. 
* **Action:** It triggers a `tool-call`.
* **Query:** `"Can I join the course if I just discovered it?"`
* **Status:** The model "pauses" here (Finish reason: `tool_calls`).

### 3. The Multi-Turn Loop (The "Agentic" Behavior)
This is where it gets interesting. Looking at your log, the model actually **called the tool 3 times.**

1.  **First Return:** Your tool responded with: `"Yes, you can join until Oct 15th."`
2.  **Second Call:** The model wasn't satisfied. It thought, *"Okay, but what are the start and end dates?"* and called `text_search` again.
3.  **Third Call:** It tried a third time with a slightly different query: `"What is the beginning and end dates of the course?"`

**Why did it do this?** Because your system prompt told it to *"Always use this tool to look up information before answering."* The model was trying to be extra thorough!

### 4. The Final Answer (Response)
Finally, after 3 tool rotations, the model decided it had enough info (or gave up on finding the specific start date).
* **Finish Reason:** `stop`.
* **Content:** It generated the text you see at the very end: *"It seems like this course starts and ends with open enrollment... until Oct 15th."*

---

### Key Terms in the JSON to Know:
| Field | Meaning |
| :--- | :--- |
| **`kind: "request"`** | Data sent **TO** the LLM (including tool results). |
| **`kind: "response"`** | Data coming **FROM** the LLM (including tool instructions). |
| **`tool_call_id`** | A unique ID (e.g., `c8djb05kt`) used to link a model's request to a specific result. |
| **`usage`** | Shows how many **tokens** you spent on that specific turn (input vs. output). |
| **`part_kind`** | Tells you if the message is `text`, a `tool-call`, or a `tool-return`. |

### Why is this useful?
If your agent ever gives a hallucinated or "weird" answer, you check `all_messages_json()` to see:
1.  Did it actually call the tool?
2.  Did the tool return the correct data?
3.  Did the model ignore the tool data?


In [44]:
import json

def debug_agent_story(json_bytes):
    # Decode bytes to string and then to list
    data = json.loads(json_bytes)
    
    
    for i, message in enumerate(data):
        kind = message.get("kind", "unknown").upper()
        parts = message.get("parts", [])
        
        for part in parts:
            pk = part.get("part_kind")
            
            if pk == "system-prompt":
                print(f"[SYSTEM INSTRUCTIONS]")
                print(f"   {part.get('content').strip()}\n")
                
            elif pk == "user-prompt":
                print(f"[USER ASKS]")
                print(f"   \"{part.get('content')}\"\n")
                
            elif pk == "tool-call":
                print(f"[TOOL CALL]: {part.get('tool_name')}")
                print(f"   Args: {part.get('args')}\n")
                
            elif pk == "tool-return":
                print(f"[TOOL RESPONSE]")
                print(f"   Result: {part.get('content')}\n")
                
            elif pk == "text":
                # Check if it's a final response or just thinking out loud
                label = "[MODEL RESPONSE]" if kind == "RESPONSE" else "💭 [THOUGHT]"
                print(f"{label}")
                print(f"   {part.get('content').strip()}\n")

        # Optional: Print token usage if it exists in this turn
        if "usage" in message and message["usage"]:
            u = message["usage"]
            print(f"(Tokens Used: {u.get('input_tokens')} in / {u.get('output_tokens')} out)")
            print("---" * 10 + "\n")

# To use it:
debug_agent_story(result.all_messages_json())

[SYSTEM INSTRUCTIONS]
   You are a helpful course assistant. 
You have access to a 'text_search' tool. 
Always use this tool to look up information before answering questions.

[USER ASKS]
   "I just discovered the course, can I join now?"

[TOOL CALL]: text_search
   Args: {"query":"Can I join the course if I just discovered it?"}

(Tokens Used: 266 in / 23 out)
------------------------------

[TOOL RESPONSE]
   Result: [{'answer': 'Yes, you can join until Oct 15th.'}]

[TOOL CALL]: text_search
   Args: {"query":"What are the start and end dates of the course?"}

(Tokens Used: 314 in / 22 out)
------------------------------

[TOOL RESPONSE]
   Result: [{'answer': 'Yes, you can join until Oct 15th.'}]

[MODEL RESPONSE]
   It appears that the start date was not in the text search results. I will use the tool again to find the start date and end date of the course.

[TOOL CALL]: text_search
   Args: {"query":"What is the beginning and end dates of the course?"}

(Tokens Used: 362 in / 54